# ATTS-RAG: GPU runs, part 3: the improved verifier (reviewer point 4)

Every job runs with verifier v3 (`ATTS_L3_V3=1`): the table-row check, the single-sentence rule for the extractive shortcut, and the longer attribution stripping. Wherever possible the **same saved answers** as the earlier runs are re-checked, so each comparison is v2 against v3 on identical drafts.

| Job | Compared with | Drafts |
|---|---|---|
| RAGTruth, small and large NLI (2,675 answers) | Part 1 RAGTruth (v2) | RAGTruth's own answers |
| Quality, 4 generators, dev + held-out | Part 1 quality (v2) | Part 1 cache (`eval/cache_snapshot/kaggle1_cache.jsonl`) |
| Llama 3, large NLI, dev + held-out | Part 1 large-NLI quality (v2) | Part 1 cache |
| Poisoning v1 (120) and v2 (64), all 11 defences | `docs/Poisoning_Study.md` (v2) | The CPU study's cache (`eval/cache_snapshot/llm_cache.jsonl`) |
| BIPIA, 5 defences (only if part 2's cache has been added) | Part 2 BIPIA (v2) | Part 2 cache (`eval/cache_snapshot/kaggle2_bipia_cache.jsonl`) |

**Settings:** *GPU T4 x2*, **Internet** *On*. **Run:** *Save Version* → *Save & Run All (Commit)*. About 2–3 hours. Download `atts_gpu_results_3.zip` from the **Output** tab.

In [ ]:
# ---- settings ------------------------------------------------------------------------------------
REPO = "https://github.com/SakthiQ/ATTS-RAG.git"
BRANCH = "claude/layer1-detection"
GENERATORS = ["llama3", "gemma4:e4b", "mistral:7b", "phi4:14b"]
NLI_LARGE = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
JUDGE1 = "qwen2.5:14b"           # scores BIPIA attacks that BIPIA scores with GPT (same judge as part 2)
JOBS = {"ragtruth": True, "quality": True, "poison": True, "bipia": True}
HELDOUT = "--qa-file eval/qa_heldout.jsonl --extra-corpus eval/corpus_heldout"
TIME_LIMIT_H = 11.0
SMOKE = False

In [ ]:
# ---- setup: code, packages, benchmarks, Ollama -------------------------------------------------------
import json, os, shutil, subprocess, sys, time, urllib.request

T0 = time.time()
BASE = "/tmp/atts"
REPO_DIR = f"{BASE}/ATTS-RAG"
RT_DATA = f"{BASE}/ragtruth/dataset"
BIPIA_DIR = f"{BASE}/bipia"
CASES = f"{BASE}/bipia_cases"
OUT = "/kaggle/working/atts_gpu3" if os.path.isdir("/kaggle/working") else f"{BASE}/out3"
R = f"{OUT}/results"
HOST = "http://127.0.0.1:11434"
for d in (BASE, OUT, R, f"{OUT}/logs"):
    os.makedirs(d, exist_ok=True)
STATUS = {}


def say(*a):
    print(f"[{(time.time() - T0) / 3600:4.1f} h]", *a, flush=True)


def sh(cmd, log=None, cwd=None, env=None):
    out = open(log, "a") if log else None
    rc = subprocess.run(cmd, shell=True, cwd=cwd, env=env, stdout=out, stderr=subprocess.STDOUT if out else None).returncode
    if out:
        out.close()
    return rc


def left_h():
    return TIME_LIMIT_H - (time.time() - T0) / 3600


sh("nvidia-smi -L")
NGPU = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 0)
assert NGPU > 0, "No GPU: set Accelerator to GPU in the notebook settings"
if not os.path.isdir(REPO_DIR):
    assert sh(f"git clone -q --depth 1 -b {BRANCH} {REPO} {REPO_DIR}") == 0, "git clone failed: is Internet on?"
COMMIT = subprocess.run(f"git -C {REPO_DIR} rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
say("ATTS-RAG commit", COMMIT, "|", NGPU, "GPU(s)")
assert os.path.exists(f"{REPO_DIR}/eval/cache_snapshot/kaggle1_cache.jsonl") and os.path.exists(f"{REPO_DIR}/app/rag/layer3/table_rows.py"), "this branch lacks the verifier v3 files: push the latest commit first"
if not os.path.isdir(RT_DATA):
    sh(f"git clone -q --depth 1 https://github.com/ParticleMedia/RAGTruth.git {BASE}/ragtruth")
if not os.path.isdir(BIPIA_DIR):
    sh(f"git clone -q --depth 1 https://github.com/microsoft/BIPIA.git {BIPIA_DIR}")

PKGS = ("langchain langchain-community langchain-huggingface langchain-chroma langchain-text-splitters langchain-ollama "
        "chromadb tiktoken rank_bm25 loguru tenacity pypdf pdfplumber pymupdf python-docx rapidocr-onnxruntime "
        "python-dotenv emoji langdetect rapidfuzz")
assert sh(f"{sys.executable} -m pip install -q -U {PKGS}", log=f"{OUT}/logs/pip.log") == 0, "pip install failed, see logs/pip.log"

if shutil.which("ollama") is None:
    sh("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw", log=f"{OUT}/logs/apt.log")
    assert sh("curl -fsSL https://ollama.com/install.sh | sh", log=f"{OUT}/logs/ollama_install.log") == 0, "Ollama install failed"
OLLAMA_ENV = dict(os.environ, OLLAMA_MODELS=f"{BASE}/ollama", OLLAMA_KEEP_ALIVE="30m", OLLAMA_HOST="127.0.0.1:11434")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=OLLAMA_ENV, stdout=open(f"{OUT}/logs/ollama_serve.log", "a"),
                               stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen(HOST + "/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(2)
say("Ollama up")

ENV = dict(os.environ, PYTHONPATH=".", HF_HOME=f"{BASE}/hf", TOKENIZERS_PARALLELISM="false",
           JUDGE_OLLAMA_HOST=HOST, OLLAMA_HOST="127.0.0.1:11434",
           ATTS_L3_V3="1")   # verifier v3 (reviewer point 4) in every job
LIM = "--limit 3" if SMOKE else ""
BIPIA_CONDITIONS = ("none (no defense)|prompt-only (documents are untrusted data)|LLM-judge chunk filter|"
                    "guard + L2 + L3 (full)|deep guard + L2 + L3 (full, deep scan)")


def save():
    json.dump({"commit": COMMIT, "smoke": SMOKE, "gpus": NGPU, "hours": round((time.time() - T0) / 3600, 2), "jobs": STATUS},
              open(f"{OUT}/status.json", "w"), indent=1)
    if os.path.isdir("/kaggle/working"):
        shutil.make_archive("/kaggle/working/atts_gpu_results_3", "zip", OUT)


def job(name, cmd, env=None, need_h=0.3):
    if left_h() < need_h:
        STATUS[name] = "skipped (time limit)"
        say("SKIP", name, "(time limit)")
        return False
    say("START", name)
    t = time.time()
    log = f"{OUT}/logs/{name}.log"
    rc = sh(cmd, log=log, cwd=REPO_DIR, env=dict(ENV, **(env or {})))
    STATUS[name] = f"{'ok' if rc == 0 else 'FAILED (exit ' + str(rc) + ')'} in {(time.time() - t) / 60:.0f} min"
    say(STATUS[name], name)
    if rc:
        print("".join(open(log, errors="replace").readlines()[-15:]))
    save()
    return rc == 0


def pull(model):
    say("pull", model)
    return sh(f"ollama pull {model}", log=f"{OUT}/logs/pull.log", env=OLLAMA_ENV) == 0


def drop(model):
    sh(f"ollama rm {model}", log=f"{OUT}/logs/pull.log", env=OLLAMA_ENV)


def tag(model):
    return model.replace(":", "-").replace(".", "")


save()

In [ ]:
# ---- 1. RAGTruth with verifier v3, small and large NLI (background, one GPU each) ----------------------------
LIM_RT = "--limit 5" if SMOKE else ""
BG = {}
if JOBS["ragtruth"]:
    for i, (name, nli) in enumerate([("ragtruth_v3_nli_small", ""), ("ragtruth_v3_nli_large", NLI_LARGE)]):
        env = dict(ENV, CUDA_VISIBLE_DEVICES=str(i % NGPU))
        if nli:
            env["NLI_MODEL"] = nli
        say("START (background)", name)
        BG[name] = (subprocess.Popen(f"python -u scripts/eval_ragtruth.py --data {RT_DATA} --out {R}/{name} --threads 2 {LIM_RT}",
                                     shell=True, cwd=REPO_DIR, env=env, stdout=open(f"{OUT}/logs/{name}.log", "a"),
                                     stderr=subprocess.STDOUT), time.time())

In [ ]:
# ---- 2. Quality with verifier v3 on part 1's drafts (no LLM calls: --offline) ---------------------------------
K1 = f"{REPO_DIR}/eval/cache_snapshot/kaggle1_cache.jsonl"
if JOBS["quality"]:
    for m in GENERATORS:
        for split, extra in (("dev", ""), ("heldout", HELDOUT)):
            job(f"quality_{split}_{tag(m)}_v3", f"python -u scripts/evaluate_pipeline.py --suite quality --model {m} --offline "
                f"--cache {K1} {extra} {LIM} --out {R}/quality_{split}_{tag(m)}_v3", need_h=0.2)
    for split, extra in (("dev", ""), ("heldout", HELDOUT)):
        job(f"quality_{split}_llama3_nli_large_v3", f"python -u scripts/evaluate_pipeline.py --suite quality --model llama3 --offline "
            f"--cache {K1} {extra} {LIM} --out {R}/quality_{split}_llama3_nli_large_v3", env={"NLI_MODEL": NLI_LARGE}, need_h=0.2)

In [ ]:
# ---- 3. Poisoning v1 + v2 with verifier v3, on the CPU study's drafts --------------------------------------
# Llama 3 is pulled so a draft missing from the cache can still be generated; the report's "LLM calls" says how many were.
CPU_CACHE = f"{OUT}/cache_cpu_study.jsonl"
if not os.path.exists(CPU_CACHE):
    shutil.copy(f"{REPO_DIR}/eval/cache_snapshot/llm_cache.jsonl", CPU_CACHE)
if JOBS["poison"]:
    pull("llama3")
    for name, pdir in (("poison_large_v1_v3", "poison_large"), ("poison_large_v2_v3", "poison_large_v2")):
        job(name, f"python -u scripts/evaluate_pipeline.py --suite poison-large --poison-dir {pdir} --extra-corpus eval/corpus_heldout "
            f"--cache {CPU_CACHE} {LIM} --out {R}/{name}", need_h=0.7)

In [ ]:
# ---- 4. BIPIA with verifier v3, on part 2's drafts (skipped unless that cache is in the repo) -------------------
K2 = f"{REPO_DIR}/eval/cache_snapshot/kaggle2_bipia_cache.jsonl"
if JOBS["bipia"] and os.path.exists(K2):
    pull("llama3")
    pull(JUDGE1)
    shutil.copy(K2, f"{OUT}/cache_bipia.jsonl")
    if job("bipia_build", f"python -u scripts/bipia_cases.py build --bipia {BIPIA_DIR} --out {CASES}", need_h=0.1):
        if job("bipia_v3_run", f"python -u scripts/evaluate_pipeline.py --suite poison-large --poison-dir {CASES} "
               f"--extra-corpus eval/corpus_heldout --conditions \"{BIPIA_CONDITIONS}\" --cache {OUT}/cache_bipia.jsonl "
               f"{LIM} --out {R}/bipia_v3", need_h=0.5):
            job("bipia_v3_score", f"python -u scripts/bipia_cases.py score --bipia {BIPIA_DIR} --cases {CASES} --results {R}/bipia_v3 "
                f"--host {HOST} --judge-model {JUDGE1}", need_h=0.2)
elif JOBS["bipia"]:
    STATUS["bipia_v3"] = "skipped (no part 2 BIPIA cache in the repo)"

In [ ]:
# ---- 5. Wait for RAGTruth, write its reports, save everything ------------------------------------------------
import glob
for name, (p, t0) in BG.items():
    while p.poll() is None and left_h() > 0.15:
        time.sleep(60)
    if p.poll() is None:
        p.terminate()
        p.wait()
        STATUS[name] = "stopped at the time limit (partial; items.jsonl is resumable)"
    else:
        STATUS[name] = f"{'ok' if p.returncode == 0 else 'FAILED (exit ' + str(p.returncode) + ')'} in {(time.time() - t0) / 60:.0f} min"
    if os.path.exists(f"{R}/{name}/items.jsonl"):
        sh(f"python -u scripts/eval_ragtruth.py --data {RT_DATA} --out {R}/{name} --report-only", log=f"{OUT}/logs/{name}.log",
           cwd=REPO_DIR, env=ENV)
save()
say("DONE")
for k, v in STATUS.items():
    print(f"  {k:36s} {v}")
for rep in sorted(glob.glob(f"{R}/*/report.md")):
    print("=" * 100, "\n", rep.replace(R + "/", ""), "\n")
    print("".join(open(rep, errors="replace").readlines()[:22]))